# Part 2, Topic 1, Lab A: Instruction Power Differences (MAIN)

---
NOTE: This lab references some (commercial) training material on [ChipWhisperer.io](https://www.ChipWhisperer.io). You can freely execute and use the lab per the open-source license (including using it in your own courses if you distribute similarly), but you must maintain notice about this source location. Consider joining our training course to enjoy the full experience.

---

**SUMMARY:** *Now that you've been introduced to the ChipWhisperer platform, we'll be using it to learn more about how the power consumed by a microcontroller varies based on what operations it is performing and what instructions it is executing.*

**LEARNING OUTCOMES:**

* Capturing a power trace with ChipWhisperer
* Making connections between various simple instructions and power traces

## Prerequisites

Hold up! Before you continue, check you've done the following tutorials:

* ☑ Jupyter Notebook Intro (you should be OK with plotting & running blocks).
* ☑ SCA101 Intro (you should have an idea of how to get hardware-specific versions running).

## Setup

This notebook runs end-to-end without hardware. Pick the back-end here:

* `VERSION = 'SIMULATED'` &rarr; replays four power traces recorded earlier on a real
  target (one per section below). This is the shipped default and produced the
  committed outputs.
* `VERSION = 'HARDWARE'` &rarr; connects to the ChipWhisperer, and each section edits
  `simpleserial-base.c`, rebuilds, re-flashes the target, and captures a live trace.

Run the next three cells and wait for `✔️ OK to continue!`.

In [1]:
SCOPETYPE = 'OPENADC'      # 'OPENADC' for CW-Lite/CW-Pro/CW-Husky, 'CWNANO' for CW-Nano
PLATFORM  = 'CWHUSKY'      # SAM4S target in the CW-Husky kit (alias for CW312_SAM4S)
CRYPTO_TARGET = 'NONE'
SS_VER = 'SS_VER_2_1'

VERSION = 'HARDWARE'      # 'SIMULATED' (pre-recorded traces) or 'HARDWARE' (live capture)

In [2]:
if VERSION == 'HARDWARE':
    %run "Lab 2_1A - Instruction Power Differences (HARDWARE).ipynb"
elif VERSION == 'SIMULATED':
    %run "Lab 2_1A - Instruction Power Differences (SIMULATED).ipynb"
else:
    raise ValueError("VERSION must be 'HARDWARE' or 'SIMULATED'")

firmware working copy: C:\Users\lewis\ChipWhisperer\chipwhisperer\firmware\mcu\simpleserial-base-lab2
INFO: Found ChipWhisperer😍
scope.gain.mode                          changed from low                       to high                     
scope.gain.gain                          changed from 0                         to 22                       
scope.gain.db                            changed from 15.0                      to 25.091743119266056       
scope.adc.samples                        changed from 131124                    to 5000                     
scope.clock.clkgen_freq                  changed from 0                         to 7363636.363636363        
scope.clock.adc_freq                     changed from 0                         to 29454545.454545453       
scope.clock.adc_rate                     changed from 0.0                       to 29454545.454545453       
scope.io.tio1                            changed from serial_tx                 to serial_rx                

In [3]:
import numpy as np
wave = capture_trace(0)      # section 0: empty trigger window
print("trace length:", len(wave))
print("✔️ OK to continue!")

trace length: 5000
✔️ OK to continue!


## "Empty" Trace

As you saw in earlier labs, the `simpleserial-base` firmware echos what we send to it. For this lab, we just want to focus on simple instructions, rather than serial communication. Open `simpleserial-base.c` in the `simpleserial-base-lab2` folder, navigate to the `get_pt()` function, and remove the `simpleserial_put()` call.

Rebuild the firmware and upload it to the target. Capture a trace, and we can plot it using the following code:

In [4]:
if VERSION == 'HARDWARE':
    # Nothing between the triggers; also drop the serial reply so the window is quiet.
    set_user_code("", keep_serial_echo=True)
    build_and_flash()
    wave = capture_trace(0)

With no user code and no serial reply, the captured window is basically the target
idling. It's our baseline - compare everything below against it.

In [5]:
cw.plot(wave)

:Curve   [x]   (y)

You shouldn't see much happening here before the target starts idling.

## Simple Instructions

To start off, let's insert some simple instructions to see if they're visible in the power trace. We're mostly looking for an instruction that executes in a single cycle, which will depend on what platform you're using for this lab. For an ARM device, try inserting the following code, which will execute 20 multiplies. It's important to mark `A` as `volatile` here to prevent the compiler from optimizing all these instructions out:

```C
	volatile long int A = 0x2BAA;
	A *= 2;
	A *= 2;
	A *= 2;
	A *= 2;
	A *= 2;
	
	A *= 2;
	A *= 2;
	A *= 2;
	A *= 2;
	A *= 2;

	A *= 2;
	A *= 2;
	A *= 2;
	A *= 2;
	A *= 2;
	
	A *= 2;
	A *= 2;
	A *= 2;
	A *= 2;
	A *= 2;
```

For the XMEGA/AVR, multiply is a more expensive instruction, so insert the following code instead:

```C

	volatile long int A = 0x2BAA;
	A += 2;
	A += 2;
	A += 2;
	A += 2;
	A += 2;
	
	A += 2;
	A += 2;
	A += 2;
	A += 2;
	A += 2;

	A += 2;
	A += 2;
	A += 2;
	A += 2;
	A += 2;
	
	A += 2;
	A += 2;
	A += 2;
	A += 2;
	A += 2;
```

Then rebuild, capture a new trace, and plot it:

In [6]:
MUL20 = "\n".join(["volatile long int A = 0x2BAA;"] + ["A *= 2;"] * 20)
ADD20 = "\n".join(["volatile long int A = 0x2BAA;"] + ["A += 2;"] * 20)

if VERSION == 'HARDWARE':
    # Multiply is single-cycle on the SAM4S/ARM; on XMEGA/AVR use ADD20 instead.
    set_user_code(ADD20 if PLATFORM in ('CW303', 'CWLITEXMEGA') else MUL20,
                  keep_serial_echo=True)
    build_and_flash()

In [7]:
wave2 = capture_trace(1)
cw.plot(wave2)

:Curve   [x]   (y)

Depending on your target, there might not be much difference here. You may want to plot both traces on the same plot:

In [8]:
cw.plot(wave) * cw.plot(wave2)

:Overlay
   .Curve.I  :Curve   [x]   (y)
   .Curve.II :Curve   [x]   (y)

## Simple Instruction Loop

You might think that it's strange that we didn't simply use a loop instead of copying the same thing out 20 times. Rectify this by replacing these instructions with a loop. **Make sure to make the loop variable volatile as well.**

In [9]:
LOOP20 = ("volatile long int A = 0x2BAA;\n"
          "volatile int i;\n"
          "for (i = 0; i < 20; i++) {\n"
          "    A *= 2;\n"
          "}")

if VERSION == 'HARDWARE':
    set_user_code(LOOP20, keep_serial_echo=True)
    build_and_flash()

In [10]:
wave3 = capture_trace(2)
cw.plot(wave3)

:Curve   [x]   (y)

Are you surprised by how different the loop is from just repeatedly performing operations? If you're comfortable with assembly, you can check the `.lss` file in the `simpleserial-base-lab2` directory. Otherwise, here's a C level overview of what (depending on how your compiler lays things out) the loop acutally looks like:

```C
i=0;
start:
    if (i > 20) {
        goto end;
    }    
    A *= 2;
    i += 1;
    goto start;
end:
    trigger_low();
    //...
```

As you can see, the microcontroller has to do a lot more than simply repeating our instruction. As such, it's something a compiler will often "unroll" (basically turn it back into our original repeated multiplies). In our case, since `i` was marked as `volatile`, the compiler avoided making this optimization. Unrolling loops also increases code size, so the compiler might also avoid unrolling when optimizing for code size.

> **What we see:** the unrolled version is one flat run of near-identical multiply
> cycles, while the loop shows a short repeating pattern - the extra compare, branch
> and counter increment on every iteration. Same 20 multiplies, noticeably more work
> (and more samples) once there's a loop around them.

## Expensive Instructions

Up to now, all the instructions we've used (besides branches in the loop) have executed in a single clock cycle. Instead, let's try an instruction that takes multiple clock cycles to execute. This will be platform specific, but for an ARM device, a divide will work well. The AVR/XMEGA doesn't actually have a divide instruction, so you'll want to use something like a multiply instead.

In [11]:
DIV_LOOP = ("volatile long int A = 0x7FFFFFFF;\n"
            "volatile int i;\n"
            "for (i = 0; i < 20; i++) {\n"
            "    A /= 3;\n"
            "}")
MUL_LOOP = DIV_LOOP.replace('A /= 3;', 'A *= 3;')

if VERSION == 'HARDWARE':
    # ARM divide takes several cycles; AVR/XMEGA has no divide, so multiply there.
    set_user_code(MUL_LOOP if PLATFORM in ('CW303', 'CWLITEXMEGA') else DIV_LOOP,
                  keep_serial_echo=True)
    build_and_flash()

In [12]:
wave4 = capture_trace(3)
cw.plot(wave4) * cw.plot(wave3)

:Overlay
   .Curve.I  :Curve   [x]   (y)
   .Curve.II :Curve   [x]   (y)

You can probably see a similar pattern to the faster instruction, but each go through the loop takes longer. We would also expect that longer instructions would also consume more power. Can you see this in your plot?

**HINT: In order to measure current, ChipWhisperer measures voltage drop across a shunt resistor. This means that the power trace is actually inverted (aka large negative swings are areas of higher power consumption).**

> **What we see:** the loop structure looks the same, but each iteration is wider -
> the divide simply takes more clock cycles than the multiply. Because ChipWhisperer
> measures voltage across a shunt, higher current shows up as a *larger negative*
> swing, so the busier divide iterations also dip further below the baseline.

## Conclusions & Next Steps

By now you should be reasonably convinced that we can gain some understanding of what a microcontroller is doing by looking at its power consumption. Going forward, we'll be focusing on more objective measurements, but what we did in this lab is still very valuable. AES implementations, for example, will often have a very distinct shape. You can use this to help identify what an unknown device is doing at different times.

Of course, there's a lot more you can do with this lab. For example, does an add look any different from an XOR? Next time, we'll be using what we learned in this lab to break a simple password check.

---
<small>NO-FUN DISCLAIMER: This material is Copyright (C) NewAE Technology Inc., 2015-2020. ChipWhisperer is a trademark of NewAE Technology Inc., claimed in all jurisdictions, and registered in at least the United States of America, European Union, and Peoples Republic of China.

Tutorials derived from our open-source work must be released under the associated open-source license, and notice of the source must be *clearly displayed*. Only original copyright holders may license or authorize other distribution - while NewAE Technology Inc. holds the copyright for many tutorials, the github repository includes community contributions which we cannot license under special terms and **must** be maintained as an open-source release. Please contact us for special permissions (where possible).

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY, FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM, OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE SOFTWARE.</small>